# Continuar o modelo de violência localmente

Este notebook **não repete o treinamento da cabeça classificadora**.

Ele:

1. carrega `models/violence/violencia_cabeca_classificadora.keras`;
2. baixa/localiza o RLVS;
3. recria os splits com a semente original;
4. executa somente o ajuste fino;
5. compara o modelo ajustado com o modelo salvo;
6. avalia no teste;
7. exporta Keras, TFLite Float32 e TFLite Float16.

Execute a partir da raiz do projeto ou usando o kernel
`Python (violence-edge-ai)`.

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import json
import random
import re
import unicodedata

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPUs TensorFlow:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

In [ ]:
def encontrar_raiz_projeto() -> Path:
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for candidato in candidatos:
        if (
            (candidato / "models").exists()
            and (candidato / "datasets").exists()
            and (candidato / "notebooks").exists()
        ):
            return candidato.resolve()

    raise FileNotFoundError(
        "Abra o Jupyter dentro da raiz do projeto violence-edge-ai."
    )


@dataclass(frozen=True)
class Configuracao:
    semente: int = 42
    tamanho_imagem: int = 160
    quantidade_quadros: int = 8
    tamanho_lote: int = 2

    epocas_ajuste: int = 3
    taxa_ajuste: float = 5e-5
    camadas_ajustaveis: int = 20
    paciencia: int = 2

    limiar: float = 0.50
    proporcao_validacao: float = 0.15
    proporcao_teste: float = 0.15


configuracao = Configuracao()
PASTA_RAIZ = encontrar_raiz_projeto()

PASTA_DADOS = PASTA_RAIZ / "datasets" / "vision" / "violence" / "rlvs"
PASTA_MODELOS = PASTA_RAIZ / "models" / "violence"
PASTA_METRICAS = PASTA_RAIZ / "outputs" / "metrics" / "violence"

PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
PASTA_METRICAS.mkdir(parents=True, exist_ok=True)

CAMINHO_CABECA = (
    PASTA_MODELOS / "violencia_cabeca_classificadora.keras"
)

if not CAMINHO_CABECA.exists():
    alternativa = PASTA_MODELOS / "melhor_modelo.keras"

    if alternativa.exists():
        CAMINHO_CABECA = alternativa
    else:
        raise FileNotFoundError(
            "Modelo não encontrado em models/violence."
        )

CAMINHO_AJUSTADO = PASTA_MODELOS / "violencia_ajuste_fino.keras"
CAMINHO_FINAL = PASTA_MODELOS / "violencia_final.keras"

random.seed(configuracao.semente)
np.random.seed(configuracao.semente)
tf.keras.utils.set_random_seed(configuracao.semente)

print("Raiz:", PASTA_RAIZ)
print("Modelo salvo:", CAMINHO_CABECA)
print(asdict(configuracao))

## Baixar ou localizar o RLVS

O download não refaz o treinamento. Ele é necessário apenas para
executar o ajuste fino e a avaliação.

In [ ]:
caminho_dataset = kagglehub.dataset_download(
    "mohamedmustafa/real-life-violence-situations-dataset",
    output_dir=str(PASTA_DADOS),
)

caminho_dataset = Path(caminho_dataset).resolve()
print("Dataset:", caminho_dataset)

In [ ]:
EXTENSOES_VIDEO = {".avi", ".mp4", ".mov", ".mkv", ".webm"}

NOMES_VIOLENCIA = {
    "violence",
    "violent",
    "fight",
    "fights",
}

NOMES_NAO_VIOLENCIA = {
    "nonviolence",
    "nonviolent",
    "nonfight",
    "normal",
    "naoviolencia",
}


def normalizar_nome(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]", "", texto.lower())


def inferir_classe(caminho: Path):
    partes = [normalizar_nome(parte) for parte in caminho.parts]

    for parte in reversed(partes):
        if parte in NOMES_NAO_VIOLENCIA:
            return 0, "nao_violencia"

        if parte in NOMES_VIOLENCIA:
            return 1, "violencia"

    return None


def video_valido(caminho: Path) -> bool:
    captura = cv2.VideoCapture(str(caminho))
    abriu = captura.isOpened()
    quantidade = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    captura.release()
    return abriu and quantidade > 0


def assinatura_rapida(caminho: Path) -> tuple[int, str]:
    tamanho = caminho.stat().st_size
    resumo = hashlib.sha256()

    with caminho.open("rb") as arquivo:
        resumo.update(arquivo.read(1024 * 1024))

        if tamanho > 1024 * 1024:
            arquivo.seek(max(0, tamanho - 1024 * 1024))
            resumo.update(arquivo.read(1024 * 1024))

    return tamanho, resumo.hexdigest()


def listar_videos(pasta_dataset: Path) -> pd.DataFrame:
    registros = []
    assinaturas_vistas = set()
    duplicados = 0

    caminhos = sorted(
        caminho
        for caminho in pasta_dataset.rglob("*")
        if caminho.suffix.lower() in EXTENSOES_VIDEO
    )

    for caminho in caminhos:
        classe = inferir_classe(caminho)

        if classe is None or not video_valido(caminho):
            continue

        assinatura = assinatura_rapida(caminho)

        if assinatura in assinaturas_vistas:
            duplicados += 1
            continue

        assinaturas_vistas.add(assinatura)
        rotulo, nome_classe = classe

        registros.append(
            {
                "caminho": str(caminho.resolve()),
                "rotulo": rotulo,
                "classe": nome_classe,
            }
        )

    tabela = pd.DataFrame(registros)

    if tabela.empty or tabela["rotulo"].nunique() != 2:
        raise RuntimeError(
            "Não foi possível encontrar as duas classes do RLVS."
        )

    print("Arquivos duplicados ignorados:", duplicados)
    return tabela


tabela_videos = listar_videos(caminho_dataset)

print("Total de vídeos únicos:", len(tabela_videos))
display(tabela_videos["classe"].value_counts().rename("quantidade"))

In [ ]:
treino_validacao, teste = train_test_split(
    tabela_videos,
    test_size=configuracao.proporcao_teste,
    random_state=configuracao.semente,
    stratify=tabela_videos["rotulo"],
)

proporcao_validacao_relativa = (
    configuracao.proporcao_validacao
    / (1.0 - configuracao.proporcao_teste)
)

treino, validacao = train_test_split(
    treino_validacao,
    test_size=proporcao_validacao_relativa,
    random_state=configuracao.semente,
    stratify=treino_validacao["rotulo"],
)

treino = treino.reset_index(drop=True)
validacao = validacao.reset_index(drop=True)
teste = teste.reset_index(drop=True)

for nome, tabela in (
    ("treino", treino),
    ("validacao", validacao),
    ("teste", teste),
):
    print(f"{nome}: {len(tabela)}")
    print(tabela["classe"].value_counts().to_dict())

    tabela.to_csv(
        PASTA_METRICAS / f"split_{nome}.csv",
        index=False,
    )

In [ ]:
def redimensionar_com_corte_central(
    quadro: np.ndarray,
    tamanho: int,
) -> np.ndarray:
    altura, largura = quadro.shape[:2]
    escala = max(tamanho / largura, tamanho / altura)

    nova_largura = max(tamanho, int(round(largura * escala)))
    nova_altura = max(tamanho, int(round(altura * escala)))

    quadro = cv2.resize(
        quadro,
        (nova_largura, nova_altura),
        interpolation=cv2.INTER_AREA,
    )

    inicio_x = (nova_largura - tamanho) // 2
    inicio_y = (nova_altura - tamanho) // 2

    return quadro[
        inicio_y : inicio_y + tamanho,
        inicio_x : inicio_x + tamanho,
    ]


def extrair_quadros(
    caminho_video: str,
    quantidade_quadros: int,
    tamanho_imagem: int,
) -> np.ndarray:
    captura = cv2.VideoCapture(caminho_video)

    if not captura.isOpened():
        raise ValueError(f"Não foi possível abrir: {caminho_video}")

    total_quadros = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_quadros <= 0:
        captura.release()
        raise ValueError(f"Vídeo sem quadros: {caminho_video}")

    indices = np.linspace(
        0,
        total_quadros - 1,
        quantidade_quadros,
        dtype=int,
    )

    quadros = []

    for indice in indices:
        captura.set(cv2.CAP_PROP_POS_FRAMES, int(indice))
        sucesso, quadro = captura.read()

        if not sucesso:
            if quadros:
                quadros.append(quadros[-1].copy())
                continue

            captura.release()
            raise ValueError(f"Falha ao ler: {caminho_video}")

        quadro = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)
        quadro = redimensionar_com_corte_central(
            quadro,
            tamanho_imagem,
        )
        quadros.append(quadro)

    captura.release()
    return np.asarray(quadros, dtype=np.float32)


def aumentar_video(video: np.ndarray) -> np.ndarray:
    if np.random.random() < 0.5:
        video = video[:, :, ::-1, :]

    brilho = np.random.uniform(0.85, 1.15)
    return np.clip(video * brilho, 0, 255)


class GeradorVideos(keras.utils.Sequence):
    def __init__(
        self,
        tabela: pd.DataFrame,
        configuracao: Configuracao,
        aumentar: bool,
        embaralhar: bool,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.tabela = tabela.reset_index(drop=True)
        self.configuracao = configuracao
        self.aumentar = aumentar
        self.embaralhar = embaralhar
        self.indices = np.arange(len(self.tabela))
        self.on_epoch_end()

    def __len__(self) -> int:
        return int(
            np.ceil(
                len(self.tabela)
                / self.configuracao.tamanho_lote
            )
        )

    def __getitem__(self, indice_lote: int):
        inicio = indice_lote * self.configuracao.tamanho_lote
        fim = min(
            inicio + self.configuracao.tamanho_lote,
            len(self.tabela),
        )

        videos = []
        rotulos = []

        for indice in self.indices[inicio:fim]:
            linha = self.tabela.iloc[indice]

            video = extrair_quadros(
                linha["caminho"],
                self.configuracao.quantidade_quadros,
                self.configuracao.tamanho_imagem,
            )

            if self.aumentar:
                video = aumentar_video(video)

            videos.append(video)
            rotulos.append(int(linha["rotulo"]))

        return (
            np.asarray(videos, dtype=np.float32),
            np.asarray(rotulos, dtype=np.float32),
        )

    def on_epoch_end(self):
        if self.embaralhar:
            np.random.shuffle(self.indices)


gerador_treino = GeradorVideos(
    treino,
    configuracao,
    aumentar=True,
    embaralhar=True,
)

gerador_validacao = GeradorVideos(
    validacao,
    configuracao,
    aumentar=False,
    embaralhar=False,
)

gerador_teste = GeradorVideos(
    teste,
    configuracao,
    aumentar=False,
    embaralhar=False,
)

print("Lotes de treino:", len(gerador_treino))
print("Lotes de validação:", len(gerador_validacao))
print("Lotes de teste:", len(gerador_teste))

## Carregar o modelo salvo e executar somente o ajuste fino

In [ ]:
def compilar_modelo(
    modelo: keras.Model,
    taxa_aprendizado: float,
):
    modelo.compile(
        optimizer=keras.optimizers.Adam(taxa_aprendizado),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="acuracia"),
            keras.metrics.Precision(name="precisao"),
            keras.metrics.Recall(name="revocacao"),
            keras.metrics.AUC(name="auc"),
        ],
        jit_compile=False,
    )


modelo_cabeca = keras.models.load_model(
    CAMINHO_CABECA,
    compile=False,
)

compilar_modelo(
    modelo_cabeca,
    configuracao.taxa_ajuste,
)

print("Entrada:", modelo_cabeca.input_shape)
print("Saída:", modelo_cabeca.output_shape)

metricas_base = modelo_cabeca.evaluate(
    gerador_validacao,
    return_dict=True,
    verbose=1,
)

print("Validação antes do ajuste:")
print(metricas_base)

In [ ]:
modelo_ajuste = keras.models.clone_model(modelo_cabeca)
modelo_ajuste.set_weights(modelo_cabeca.get_weights())

extrator = modelo_ajuste.get_layer(
    "extrator_quadros"
).layer

extrator.trainable = True

for camada in extrator.layers:
    camada.trainable = False

for camada in extrator.layers[
    -configuracao.camadas_ajustaveis:
]:
    if not isinstance(camada, layers.BatchNormalization):
        camada.trainable = True

compilar_modelo(
    modelo_ajuste,
    configuracao.taxa_ajuste,
)

treinaveis = sum(
    int(np.prod(peso.shape))
    for peso in modelo_ajuste.trainable_weights
)

print("Parâmetros treináveis:", f"{treinaveis:,}".replace(",", "."))

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        CAMINHO_AJUSTADO,
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=configuracao.paciencia,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        PASTA_METRICAS / "historico_ajuste_fino.csv",
    ),
]

historico_ajuste = modelo_ajuste.fit(
    gerador_treino,
    validation_data=gerador_validacao,
    epochs=configuracao.epocas_ajuste,
    callbacks=callbacks,
)

In [ ]:
candidato = keras.models.load_model(
    CAMINHO_AJUSTADO,
    compile=False,
)

compilar_modelo(
    candidato,
    configuracao.taxa_ajuste,
)

metricas_ajuste = candidato.evaluate(
    gerador_validacao,
    return_dict=True,
    verbose=1,
)

print("Antes:", metricas_base)
print("Depois:", metricas_ajuste)

if metricas_ajuste["auc"] >= metricas_base["auc"]:
    modelo_final = candidato
    origem_final = "ajuste_fino"
else:
    modelo_final = modelo_cabeca
    origem_final = "cabeca_classificadora"

modelo_final.save(CAMINHO_FINAL)

print("Modelo escolhido:", origem_final)
print("Salvo em:", CAMINHO_FINAL)

## Avaliar no teste

In [ ]:
probabilidades = modelo_final.predict(
    gerador_teste,
    verbose=1,
).reshape(-1)

rotulos_reais = teste["rotulo"].to_numpy(dtype=int)

rotulos_previstos = (
    probabilidades >= configuracao.limiar
).astype(int)

relatorio_texto = classification_report(
    rotulos_reais,
    rotulos_previstos,
    target_names=["nao_violencia", "violencia"],
    zero_division=0,
)

print(relatorio_texto)

matriz = confusion_matrix(
    rotulos_reais,
    rotulos_previstos,
)

ConfusionMatrixDisplay(
    matriz,
    display_labels=["nao_violencia", "violencia"],
).plot()

plt.title("Matriz de confusão — teste")
plt.show()

predicoes = teste.copy()
predicoes["probabilidade_violencia"] = probabilidades
predicoes["predicao"] = rotulos_previstos
predicoes.to_csv(
    PASTA_METRICAS / "predicoes_teste.csv",
    index=False,
)

## Exportar e validar TFLite

In [ ]:
caminhos_tflite = {}

for nome, configurar in (
    ("float32", None),
    ("float16", "float16"),
):
    destino = (
        PASTA_MODELOS
        / "tflite"
        / f"violencia_{nome}.tflite"
    )
    destino.parent.mkdir(parents=True, exist_ok=True)

    conversor = tf.lite.TFLiteConverter.from_keras_model(
        modelo_final
    )

    if configurar == "float16":
        conversor.optimizations = [tf.lite.Optimize.DEFAULT]
        conversor.target_spec.supported_types = [tf.float16]

    conteudo = conversor.convert()
    destino.write_bytes(conteudo)
    caminhos_tflite[nome] = destino

    interpretador = tf.lite.Interpreter(
        model_path=str(destino)
    )
    interpretador.allocate_tensors()

    entrada = interpretador.get_input_details()[0]
    saida = interpretador.get_output_details()[0]

    print(
        nome,
        destino,
        f"{destino.stat().st_size / 1024**2:.2f} MB",
    )
    print("Entrada:", entrada["shape"], entrada["dtype"])
    print("Saída:", saida["shape"], saida["dtype"])

In [ ]:
resultado = {
    "configuracao": asdict(configuracao),
    "modelo_inicial": str(CAMINHO_CABECA),
    "modelo_escolhido": origem_final,
    "modelo_final": str(CAMINHO_FINAL),
    "quantidade_videos": {
        "treino": len(treino),
        "validacao": len(validacao),
        "teste": len(teste),
    },
    "metricas_validacao_antes": {
        chave: float(valor)
        for chave, valor in metricas_base.items()
    },
    "metricas_validacao_depois": {
        chave: float(valor)
        for chave, valor in metricas_ajuste.items()
    },
    "matriz_confusao_teste": matriz.tolist(),
    "relatorio_teste": classification_report(
        rotulos_reais,
        rotulos_previstos,
        target_names=["nao_violencia", "violencia"],
        output_dict=True,
        zero_division=0,
    ),
    "tflite": {
        chave: str(valor)
        for chave, valor in caminhos_tflite.items()
    },
}

caminho_resultado = (
    PASTA_METRICAS / "resultado_continuacao.json"
)

caminho_resultado.write_text(
    json.dumps(
        resultado,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Concluído.")
print("Modelo final:", CAMINHO_FINAL)
print("Métricas:", caminho_resultado)